# GBM Grid Search — US Equities Panel

With 3,199 stocks and 72 features, the question is whether gradient boosting
can capture non-linear feature interactions that linear models miss. On daily
returns, the linear baseline suggests the cross-sectional
relationship is largely linear. The GBM grid (5 leaf profiles × 3 loss
functions = 15 configurations) tests whether tree-based non-linearity adds
value on this broad, noisy panel.

**Learning Objectives**:
- Test whether non-linear models improve on the linear baseline for broad panels
- Compare loss functions (MSE vs MAE vs Huber) on noisy daily returns
- Track IC learning curves to identify overfitting on 3,199-stock cross-sections
- Determine if horizon matters: daily vs weekly vs monthly GBM performance

**Book Reference**: Chapter 12, Section 12.2 (GBM Libraries)

**Prerequisites**: `03_financial_features.py`, `04_temporal.py`, [`05_evaluation`](05_evaluation.ipynb)

In [1]:
"""GBM Grid Search — config-driven regularization profiles × loss functions."""

import warnings

import numpy as np
import polars as pl
import yaml

from case_studies.utils.gbm import (
    prepare_gbm_folds,
    register_gbm_result,
    train_gbm_config,
)
from case_studies.utils.registry import (
    build_training_spec,
    get_training_dir,
    load_prediction_metrics,
    load_prediction_sets,
    training_hash_from_spec,
    training_run_status,
)
from utils.modeling import load_configs, load_modeling_dataset
from utils.paths import get_case_study_dir

warnings.filterwarnings("ignore")

In [2]:
CASE_STUDY_ID = "us_equities_panel"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
MAX_FOLDS = 0
FORCE_RETRAIN = False  # Set True to retrain configs that already have complete hashes
PREDICTION_SPLIT = "validation"
TRAIN_SAMPLE_FRAC = 1.0  # <1.0 subsamples training rows per fold (val is never sampled). Use for memory-constrained runs on large datasets.

In [3]:
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((CASE_DIR / "config" / "setup.yaml").read_text())

if not PRIMARY_LABEL:
    PRIMARY_LABEL = setup["labels"]["primary"]

# Device: read from setup.yaml, fall back to GPU detection
gbm_config = setup.get("modeling", {}).get("gbm", {})
DEVICE = gbm_config.get("device", "cuda")
MAX_BIN = 63  # GPU default
import torch

if DEVICE != "cpu" and not torch.cuda.is_available():
    DEVICE, MAX_BIN = "cpu", 255

print(f"Case study: {CASE_STUDY_ID} | Device: {DEVICE} | max_bin: {MAX_BIN}")

Case study: us_equities_panel | Device: cpu | max_bin: 63


## 1. Load Data and Model Configs

GBM configs are defined in `config/training/{label}.yaml` under the `gbm:` key.
Each config references a preset in `config/lgb/` with the complete
LightGBM parameter set. To modify the grid, edit the label config file.

In [4]:
mds = load_modeling_dataset(CASE_STUDY_ID, PRIMARY_LABEL, max_symbols=MAX_SYMBOLS)

dataset = mds.dataset
feature_names = mds.feature_names
label_col = mds.label_col
date_col = mds.date_col
entity_col = mds.entity_cols[0] if mds.entity_cols else "symbol"
splits = mds.splits[: MAX_FOLDS or None]

print(f"Dataset: {len(dataset):,} rows × {len(feature_names)} features")
print(f"Label: {label_col} | Task: {mds.task_type} | Folds: {len(splits)}")

Dataset: 9,205,450 rows × 72 features
Label: fwd_ret_1d | Task: regression | Folds: 16


In [5]:
configs = load_configs(CASE_STUDY_ID, PRIMARY_LABEL, family="gbm")

print(f"\n{len(configs)} configs × {len(splits)} folds\n")
for cfg in configs:
    leaves = cfg["params"].get("num_leaves", 31)
    obj = cfg["params"].get("objective", "regression")
    n_trees = cfg.get("max_iterations", 500)
    print(f"  {cfg['config_name']:25s}  leaves={leaves:3d}  obj={obj}  trees={n_trees}")


15 configs × 16 folds

  default_mse                leaves= 31  obj=regression  trees=500
  default_mae                leaves= 31  obj=regression_l1  trees=500
  default_huber              leaves= 31  obj=huber  trees=500
  leaves_7_mse               leaves=  7  obj=regression  trees=500
  leaves_7_mae               leaves=  7  obj=regression_l1  trees=500
  leaves_7_huber             leaves=  7  obj=huber  trees=500
  leaves_15_mse              leaves= 15  obj=regression  trees=500
  leaves_15_mae              leaves= 15  obj=regression_l1  trees=500
  leaves_15_huber            leaves= 15  obj=huber  trees=500
  leaves_31_mse              leaves= 31  obj=regression  trees=500
  leaves_31_mae              leaves= 31  obj=regression_l1  trees=500
  leaves_31_huber            leaves= 31  obj=huber  trees=500
  leaves_63_mse              leaves= 63  obj=regression  trees=500
  leaves_63_mae              leaves= 63  obj=regression_l1  trees=500
  leaves_63_huber            leaves= 63  ob

## 2. Prepare CV Folds

GBM folds use float32 (LightGBM native precision) and skip
imputation/scaling — gradient boosting handles missing values natively.

In [6]:
dataset_pd = dataset.to_pandas()
fold_data = prepare_gbm_folds(
    dataset_pd,
    splits,
    feature_names,
    label_col,
    date_col,
    entity_col,
    task_type=mds.task_type,
    class_values=mds.class_values,
    temporal_by_fold=mds.temporal_by_fold,
    temporal_keys=mds.temporal_keys,
    temporal_feature_names=mds.temporal_feature_names,
    train_sample_frac=TRAIN_SAMPLE_FRAC,
)

for f in fold_data:
    print(f"  Fold {f['fold']}: train={f['n_train']:,}  val={f['n_val']:,}")

  Fold 0: train=4,922,932  val=614,597
  Fold 1: train=4,638,284  val=636,426
  Fold 2: train=4,348,457  val=580,970
  Fold 3: train=4,100,158  val=513,206
  Fold 4: train=3,836,956  val=515,275
  Fold 5: train=3,587,626  val=480,539
  Fold 6: train=3,360,690  val=427,476
  Fold 7: train=3,082,162  val=463,753
  Fold 8: train=2,763,661  val=481,589
  Fold 9: train=2,458,039  val=436,748
  Fold 10: train=2,168,769  val=389,494
  Fold 11: train=1,898,335  val=351,768
  Fold 12: train=1,682,775  val=290,793
  Fold 13: train=1,474,649  val=264,735
  Fold 14: train=1,266,587  val=252,072
  Fold 15: train=1,063,065  val=231,016


## 3. Train All Configs

For each config, train one LightGBM model per fold to `max_iterations` trees.
Cross-sectional IC is evaluated at checkpoints (every 50 iterations) to
detect overfitting — configs that peak early and decay indicate too much capacity.

In [7]:
results = []
for cfg in configs:
    # Pre-compute registry training dir so boosters go directly there
    spec = build_training_spec(
        cfg["family"],
        cfg["config_name"],
        label_col,
        n_folds=len(fold_data),
        max_bin=MAX_BIN,
        checkpoint_interval=cfg.get("checkpoint_interval", 50),
        train_sample_frac=TRAIN_SAMPLE_FRAC,
    )
    train_dir = get_training_dir(CASE_STUDY_ID, spec)

    # Skip if this config's hash is already complete (unless FORCE_RETRAIN)
    _status = training_run_status(CASE_STUDY_ID, spec)
    _training_hash = training_hash_from_spec(spec)
    _split_rows = load_prediction_sets(
        CASE_STUDY_ID,
        training_hash=_training_hash,
        split=PREDICTION_SPLIT,
    )
    _split_complete = not _split_rows.is_empty()
    if _status.complete and _split_complete and not FORCE_RETRAIN:
        # Already trained + registered: rebuild a minimal result from the
        # registry so the grid + learning-curve sections render on a
        # fully-cached checkout. (A bare `continue` here drops the config
        # from `results`, printing an empty grid when every config is
        # registered.) best_ic is the authoritative registered value;
        # best_iter and the curves come from learning_curves.parquet.
        _pred_hash = _split_rows["prediction_hash"][0]
        _metrics = load_prediction_metrics(CASE_STUDY_ID, prediction_hash=_pred_hash)
        _best_ic = float(_metrics["ic_mean"][0]) if not _metrics.is_empty() else float("nan")
        _curves = []
        _lc_path = train_dir / "learning_curves.parquet"
        if _lc_path.exists():
            _curves = pl.read_parquet(_lc_path).to_dicts()
        _best_iter = 0
        if _curves:
            _best_iter = int(max(_curves, key=lambda c: c["ic_mean"])["iteration"])
        print(
            f"  {cfg['config_name']:25s}  iter={_best_iter:4d}  IC={_best_ic:+.4f}  "
            f"(cached, {_status.summary()})"
        )
        results.append(
            {
                "config_name": cfg["config_name"],
                "best_iter": _best_iter,
                "best_ic": _best_ic,
                "elapsed_s": 0.0,
                "learning_curves": _curves,
                "cached": True,
            }
        )
        continue
    if _status.complete and not _split_complete:
        print(f"  {cfg['config_name']:25s}  RETRAIN — missing {PREDICTION_SPLIT} predictions")
    elif _status.partial:
        print(f"  {cfg['config_name']:25s}  RETRAIN — partial state: {_status.summary()}")

    result = train_gbm_config(
        cfg,
        fold_data,
        feature_names=feature_names,
        device=DEVICE,
        max_bin=MAX_BIN,
        entity_col=entity_col,
        date_col=date_col,
        task_type=mds.task_type,
        class_values=mds.class_values,
        save_dir=train_dir,
    )
    results.append(result)
    print(
        f"  {result['config_name']:25s}  iter={result['best_iter']:4d}  "
        f"IC={result['best_ic']:+.4f}  ({result['elapsed_s']:.0f}s)"
    )

    # Register immediately after training — incremental save protects against
    # interruption losing work on large sweeps.
    register_gbm_result(
        CASE_STUDY_ID,
        result,
        cfg,
        label_col,
        n_folds=len(fold_data),
        max_bin=MAX_BIN,
        entry_point="07_gbm",
        date_col=date_col,
        entity_col=entity_col,
        train_sample_frac=TRAIN_SAMPLE_FRAC,
        prediction_split=PREDICTION_SPLIT,
    )

  default_mse                iter= 500  IC=+0.0219  (cached, complete (hash=c1df40085fc0))
  default_mae                iter= 500  IC=+0.0304  (cached, complete (hash=4b142455e6fb))
  default_huber              iter= 500  IC=+0.0228  (cached, complete (hash=965fbe33a2b9))
  leaves_7_mse               iter= 500  IC=+0.0200  (cached, complete (hash=9264139f37b5))
  leaves_7_mae               iter= 500  IC=+0.0250  (cached, complete (hash=0d7c7cab296b))
  leaves_7_huber             iter= 500  IC=+0.0195  (cached, complete (hash=e22cbcd8680d))
  leaves_15_mse              iter= 500  IC=+0.0211  (cached, complete (hash=8523f8f286b9))
  leaves_15_mae              iter= 500  IC=+0.0284  (cached, complete (hash=b782932d8525))
  leaves_15_huber            iter= 500  IC=+0.0208  (cached, complete (hash=92c9c6f958c1))
  leaves_31_mse              iter= 500  IC=+0.0221  (cached, complete (hash=c09051b3253f))
  leaves_31_mae              iter= 500  IC=+0.0305  (cached, complete (hash=3d92e74fb46d))

## 4. Grid Results

All configs ranked by peak IC (best checkpoint). The best config × iteration
combination is selected for downstream prediction generation.

In [8]:
results.sort(key=lambda r: r["best_ic"], reverse=True)
best = results[0] if results else None

print(f"{'Config':25s}  {'Iter':>5s}  {'IC':>8s}  {'Time':>6s}")
print("-" * 50)
for r in results:
    marker = " *" if r is best else ""
    print(
        f"  {r['config_name']:25s}  {r['best_iter']:5d}  {r['best_ic']:+.4f}  {r['elapsed_s']:5.0f}s{marker}"
    )

if best:
    print(f"\nBest: {best['config_name']} @ {best['best_iter']} trees (IC={best['best_ic']:+.4f})")

Config                      Iter        IC    Time
--------------------------------------------------
  leaves_63_mae                500  +0.0318      0s *
  leaves_31_mae                500  +0.0305      0s
  default_mae                  500  +0.0304      0s
  leaves_15_mae                500  +0.0284      0s
  leaves_7_mae                 500  +0.0250      0s
  leaves_63_mse                500  +0.0236      0s
  leaves_63_huber              500  +0.0230      0s
  default_huber                500  +0.0228      0s
  leaves_31_huber              500  +0.0221      0s
  leaves_31_mse                500  +0.0221      0s
  default_mse                  500  +0.0219      0s
  leaves_15_mse                500  +0.0211      0s
  leaves_15_huber              500  +0.0208      0s
  leaves_7_mse                 500  +0.0200      0s
  leaves_7_huber               500  +0.0195      0s

Best: leaves_63_mae @ 500 trees (IC=+0.0318)


## 5. Learning Curves

IC at checkpoints (every 50 iterations) for each config. Configs that peak
early and decay indicate overfitting; those that plateau show good regularization.

In [9]:
all_curves = pl.DataFrame([c for r in results for c in r["learning_curves"]])
if all_curves.height > 0:
    checkpoints = sorted(all_curves["iteration"].unique().to_list())
    display_cps = [cp for cp in [50, 100, 200, 300, 500] if cp in checkpoints]

    print(f"{'Config':25s}", end="")
    for cp in display_cps:
        print(f" {cp:>7d}", end="")
    print()

    for r in results:
        cfg_data = all_curves.filter(pl.col("config") == r["config_name"])
        print(f"  {r['config_name']:25s}", end="")
        for cp in display_cps:
            row = cfg_data.filter(pl.col("iteration") == cp)
            if row.height > 0:
                print(f" {row['ic_mean'][0]:+7.4f}", end="")
            else:
                print(f" {'N/A':>7s}", end="")
        print()

Config                         50     100     200     300     500
  leaves_63_mae             +0.0228 +0.0256 +0.0284 +0.0298 +0.0318
  leaves_31_mae             +0.0215 +0.0238 +0.0264 +0.0285 +0.0305
  default_mae               +0.0217 +0.0235 +0.0269 +0.0285 +0.0304
  leaves_15_mae             +0.0215 +0.0230 +0.0246 +0.0259 +0.0284
  leaves_7_mae              +0.0217 +0.0204 +0.0220 +0.0236 +0.0250
  leaves_63_mse             +0.0153 +0.0181 +0.0207 +0.0215 +0.0236
  leaves_63_huber           +0.0151 +0.0175 +0.0209 +0.0216 +0.0229
  default_huber             +0.0163 +0.0181 +0.0199 +0.0211 +0.0228
  leaves_31_huber           +0.0139 +0.0172 +0.0194 +0.0207 +0.0221
  leaves_31_mse             +0.0133 +0.0162 +0.0192 +0.0205 +0.0221
  default_mse               +0.0152 +0.0166 +0.0195 +0.0208 +0.0219
  leaves_15_mse             +0.0118 +0.0154 +0.0176 +0.0188 +0.0211
  leaves_15_huber           +0.0114 +0.0148 +0.0180 +0.0191 +0.0208
  leaves_7_mse              +0.0095 +0.0140 +0.016

## 6. Registration Complete

Each config was registered immediately after training (see Section 3).
This protects against interruption — all completed configs are already
persisted in `run_log/registry.db`.

In [10]:
print(f"All {len(results)} configs registered.")

All 15 configs registered.


## 7. Key Takeaways

GBM adds almost nothing on daily returns (nearly matching linear), but the gap
opens at longer horizons where GBM clearly achieves a higher IC than linear.
For the broadest equity panel, the daily cross-section is essentially linear --- the
Fundamental Law's breadth advantage doesn't require non-linear modeling. MAE
loss consistently achieves a higher IC than MSE, a practical lesson for noisy return
prediction: robust loss functions matter more than model complexity.

**Next**: [`08_tabular_dl`](08_tabular_dl.ipynb) and the DL notebooks test whether attention
mechanisms or temporal architectures add value on this panel.